# Stage 02 — Chunking

**Track A (Buse) · Stage 2 of 10**

| | |
|---|---|
| **Input** | `data/interim/pages.jsonl` |
| **Output** | `data/processed/chunks.jsonl` matching the `Chunk` model in `contracts/retrieval_J.py` |
| **Promotes to** | `src/research_assistant/ingestion/chunk_B.py` |
| **Config** | `configs/ingestion_B.yaml` → `chunk:` |

## Why this is the highest-leverage stage in Track A

Chunking sets the ceiling on everything after it. A fact split across two chunks can
never be retrieved as one hit, no matter how good the embedding model or the reranker
is. In most RAG projects the biggest single metric jump comes from here, not from the
model swaps that get more attention.

This is the stage to spend your tactic-survey budget on.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Strategy | `section_aware` | `fixed_window`, `recursive_character`, `semantic`, `proposition` | Papers have real section structure and it is free signal. Fixed windows cut mid-argument. Semantic chunking costs an embedding pass per sentence and mostly rediscovers paragraph breaks on this document type. |
| Target size | 350 tokens | 200 / 512 / 800 | Roughly one dense paragraph. Small enough that a hit is precise, large enough to contain a claim plus its qualifier. Sweep it below before trusting this. |
| Overlap | 60 tokens | 0, 100+ | Cheap insurance against a claim landing on a boundary. Above ~20 percent overlap you mostly inflate the index and duplicate hits in the candidate set. |
| Context prefix | `TITLE > SECTION` prepended | Nothing, LLM-written context | A chunk that says "we improve on this by 4 points" is unretrievable without knowing what "this" is. The cheap version of contextual retrieval. |
| Merge tiny chunks | Yes, under 80 tokens | Keep | Sub-80-token chunks are usually captions or stray headings. They pollute BM25 and win on short queries for the wrong reasons. |

**The sweep below is the point of this notebook.** Do not accept 350 tokens because
it is written above. Run the sweep after stage 05 exists and let the numbers choose.

In [ ]:
from _nbsetup_B import REPO, DATA, load_cfg, resolve, ensure_dirs
import json, itertools
from pathlib import Path

cfg = load_cfg("ingestion")
cfg

In [ ]:
import tiktoken
enc = tiktoken.get_encoding(cfg["chunk"]["tokenizer"])

def ntok(s):
    return len(enc.encode(s))

pages = [json.loads(l) for l in (resolve(cfg["corpus"]["interim_dir"]) / "pages.jsonl")
         .read_text(encoding="utf-8").splitlines() if l.strip()]
print(len(pages), "pages")

def group_sections(pages):
    # Consecutive pages sharing a section become one unit, so a section that spans
    # a page break is not cut at the break.
    units, cur = [], None
    for p in pages:
        key = (p["paper_id"], p["section"])
        if cur and cur["key"] == key:
            cur["text"] += "\n" + p["text"]
            cur["page_end"] = p["page"]
        else:
            if cur:
                units.append(cur)
            cur = dict(key=key, paper_id=p["paper_id"], section=p["section"],
                       text=p["text"], page_start=p["page"], page_end=p["page"])
    if cur:
        units.append(cur)
    return units

units = group_sections(pages)
print(len(units), "section units")

In [ ]:
import re, hashlib

def split_paragraphs(text):
    parts = [p.strip() for p in re.split(r"\n\s*\n|\n(?=[A-Z])", text) if p.strip()]
    return parts or [text]

def chunk_unit(unit, target, overlap, min_tok, max_tok):
    paras, out, buf, buf_tok = split_paragraphs(unit["text"]), [], [], 0
    for para in paras:
        t = ntok(para)
        if buf_tok + t > target and buf:
            out.append(" ".join(buf))
            # carry the tail of the previous chunk forward as overlap
            tail, tail_tok = [], 0
            for prev in reversed(buf):
                pt = ntok(prev)
                if tail_tok + pt > overlap:
                    break
                tail.insert(0, prev); tail_tok += pt
            buf, buf_tok = tail, tail_tok
        buf.append(para); buf_tok += t
        if buf_tok >= max_tok:
            out.append(" ".join(buf)); buf, buf_tok = [], 0
    if buf:
        out.append(" ".join(buf))
    # merge anything below the floor into its neighbour
    merged = []
    for c in out:
        if merged and ntok(c) < min_tok:
            merged[-1] = merged[-1] + " " + c
        else:
            merged.append(c)
    return merged

In [ ]:
titles = {}
mpath = resolve(cfg["corpus"]["manifest"])
if mpath.exists():
    for line in mpath.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line); titles[r["paper_id"]] = r.get("title") or ""

def build_chunks(units, c):
    rows = []
    for u in units:
        for i, text in enumerate(chunk_unit(u, c["target_tokens"], c["overlap_tokens"],
                                            c["min_tokens"], c["max_tokens"])):
            title = titles.get(u["paper_id"], "")
            body = f"{title} > {u['section']}\n{text}" if c["prepend_context"] else text
            rows.append(dict(
                chunk_id=hashlib.sha1(f"{u['paper_id']}|{u['section']}|{i}".encode()).hexdigest()[:16],
                paper_id=u["paper_id"], title=title, section=u["section"],
                page_start=u["page_start"], page_end=u["page_end"],
                text=body, n_tokens=ntok(body),
            ))
    return rows

chunks = build_chunks(units, cfg["chunk"])
import pandas as pd
df = pd.DataFrame(chunks)
print(len(df), "chunks")
df["n_tokens"].describe()

### The sweep

Run this **after stage 05 exists**. Until you have a labelled query set, chunk-size
choice is guesswork with a nice-looking histogram attached.

Record the winning configuration and the delta in `docs/tactics_ledger_B.md`. The
comparison you care about is nDCG@5 and recall@20, not chunk count.

In [ ]:
# Sweep skeleton. Fill the scoring call once notebook 06 exposes `evaluate(chunks)`.
SWEEP = [
    dict(target_tokens=200, overlap_tokens=40),
    dict(target_tokens=350, overlap_tokens=60),
    dict(target_tokens=512, overlap_tokens=80),
    dict(target_tokens=800, overlap_tokens=100),
]

results = []
for s in SWEEP:
    c = {**cfg["chunk"], **s}
    ch = build_chunks(units, c)
    results.append(dict(**s, n_chunks=len(ch),
                        mean_tokens=round(sum(x["n_tokens"] for x in ch) / len(ch), 1),
                        ndcg5=None))  # <- from notebook 06
pd.DataFrame(results)

In [ ]:
processed = resolve(cfg["corpus"]["processed_dir"]); ensure_dirs(processed)
out = processed / "chunks.jsonl"
with out.open("w", encoding="utf-8") as f:
    for r in chunks:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote", len(chunks), "chunks to", out)

required = cfg["metadata_schema"]["required"]
missing = [k for k in required if k not in chunks[0]]
assert not missing, f"contract violation, missing fields: {missing}"
print("schema ok:", required)

## Exit checks

- [ ] No chunk exceeds `max_tokens`, none falls below `min_tokens`.
- [ ] Every required field from `metadata_schema` is present. This is the contract
      Sude's tools read, so a missing field breaks her side, not yours.
- [ ] Read five random chunks. Each should be understandable standing alone. If you
      cannot tell what a chunk is about, the context prefix is not doing its job.
- [ ] Chunk count is roughly 30-60 per paper. Far more means the section detector in
      stage 01 is over-firing.

## Promote to `src/`

`chunk_B.py` gets `split_paragraphs`, `chunk_unit`, `build_chunks`. The sweep stays
in the notebook, because it is evidence for the report, not production code.